In [7]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_charge as yuc

yu.setpath('analysis_sgm')
enss=['b','c','d','e']

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
symmetrizeQ=True
[ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_c2pt')

ens2data={}; ens2Njk={}
for ens in enss:
    path=f'/p/project1/ngff/li47/code/projectData/07_Nsgm/charges/{yu.ens2full[ens]}.h5'
    [c2pt_disc,tfs_conn,tfs_disc,tf2c2pt_conn,key2tf2c3pt,key2tf2ratio]=yuc.load(path,symmetrizeQ=symmetrizeQ)
    ens2data[ens]=[c2pt_disc,tfs_conn,tfs_disc,tf2c2pt_conn,key2tf2c3pt,key2tf2ratio]
    ens2Njk[ens]=len(c2pt_disc)

ens2Njk

{'b': 725, 'c': 400, 'd': 493, 'e': 445}

In [9]:
path='data_aux/sgm_ChPT.csv'
df=pd.read_csv(path, index_col=0)
def func(ele):
    m,e=ele.split(',')
    m=m.split('(')[-1]
    e=e.split(')')[0]
    return float(m),float(e)

dic_sgm_ChPT={(row_label, col_label): func(df.loc[row_label, col_label])
    for row_label in df.index
    for col_label in df.columns
}

In [10]:
g='gS+'
ds=1

app = {1:'',2:'_ds2'}[ds]

ens2dics={}
for ens in enss:
    pars_jk_meff2st=ens2pars_jk_meff2st[ens]
    [c2pt_disc,tfs_conn,tfs_disc,tf2c2pt_conn,key2tf2c3pt,key2tf2ratio]=ens2data[ens]
    tf2ratio=key2tf2ratio[g]
    
    tfmins_2st=tfs_conn[:-3]
    # tcmins_2st=np.arange(1,int(0.5/yu.ens2a[ens]),int(0.2/yu.ens2a[ens]))
    tcmins_2st=np.arange(1,round(0.45/yu.ens2a[ens]))
    
    label2fits={}
    for label in ['2st2step_SYM','2st2step_SYMshare','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']:
        label2fits[label]=yu.doFits_3pt(label,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,downSampling=[1,ds],label=f'{g}_{ens}_{label}{app}',verbose=1)
        
    # n=int(0.2/yu.ens2a[ens])
    n=2
    def dE2lbd(dE):
        lbd0=dE*n
        return np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    
    def lbd2tf2ratio(dE):    
        lbd=dE2lbd(dE)
        
        tf2ratio={}
        for tf in tfs_conn:
            c3=key2tf2c3pt[g][tf]
            c3=-(np.roll(c3,-n,axis=-1)+np.roll(c3,n,axis=-1)-2*c3) + lbd**2*c3
            c2=(lbd**2)*c2pt_disc[:,tf]
            tf2ratio[tf]=c3/c2[:,None]
        return tf2ratio
    tfmins=tfmins_2st
    tcmins=[n+tcmin for tcmin in tcmins_2st]
    
    fits_laplace=yu.doFits_3pt_lbd(lbd2tf2ratio,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'{g}_{ens}_lbd{app}',verbose=1,downSampling=[1,ds],overwrite=False)
    fits_laplace=[[(tfmin,tcmin),np.array([pars_jk[:,0],np.abs(pars_jk[:,1])]).T,chi2_jk,Ndof] for (tfmin,tcmin),pars_jk,chi2_jk,Ndof in fits_laplace]

    fitlabel_chosen=(8,n+2)
    # fitlabel_chosen=fits_laplace_ds2[0][0]
    fit_MA_laplace=yu.doMA_3pt(fits_laplace,fitlabels=fitlabel_chosen)
    print(ens,fitlabel_chosen,yu.jackme_un2str(fit_MA_laplace[0][:,1]*yu.ens2aInv[ens]))

    dE=np.mean(fit_MA_laplace[0][:,1])
    tf2ratio_laplace=lbd2tf2ratio(dE)
    # fits_const_2=yu.doFits_3pt('const',tf2ratio_laplace,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'const_2_laplace'+extraLabel)
    # fit_const_MA_2=yu.doMA_3pt(fits_const_2,fitlabels=fitlabel_chosen)

    xunit=yu.ens2a[ens]
    yunit=yu.ens2amul_iso[ens]*yu.ens2aInv[ens]
    
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,None,None,None,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['None'],
    }
    dic_lbd={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio_laplace,None,fits_laplace,None,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,n+1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['white'],
    }
    
    label2dic={}
    for label in label2fits.keys():
        label2dic[label]={
            'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,None,label2fits[label]],
            'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
            'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,5,None,None],
            'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
            'xyunit':[xunit,yunit],
            'mfc:[global]':['None'],
        }
    
    ens2dics[ens]=[dic,dic_lbd,label2dic]

colHeaders=['ratio','fit_laplace','2st2step_SYM','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']
fig,axs=yu.makePlot_3pt([ens2dics[ens][0] for ens in enss],shows=['rainbow'],colHeaders=colHeaders,fontsize_colHeaders=20, sharey=True)
yu.addRowHeader(axs,[yu.ens2label[ens] for ens in enss])
axs[0,0].set_ylim([0,80])
fig,axs=yu.makePlot_3pt([ens2dics[ens][1] for ens in enss],shows=['rainbow']+[None]*(colHeaders.index('fit_laplace')-1)+['fit_const'],figAxs=(fig,axs),colHeaders=None)

for i,label in enumerate(colHeaders):
    if label in label2dic.keys(): 
        fig,axs=yu.makePlot_3pt([ens2dics[ens][2][label] for ens in enss],shows=['rainbow']+[None]*(i-1)+['fit_2st'],figAxs=(fig,axs),colHeaders=None)
fig,axs=yu.makePlot_3pt([ens2dics[ens][2]['2st2step_SYMshare'] for ens in enss],shows=['rainbow','fit_2st'],figAxs=(fig,axs),colHeaders=None)
        
yu.finalizePlot(f'rainbow_fits{app}')

b (8, 4) 337(26)
c (8, 4) 372(30)
d (8, 4) 370(26)
e (8, 4) 446(39)


In [15]:
ylim=[0,80]

tfphys=[0.4,0.5,0.6,0.7,0.8,0.9]
tcphys=[0.1,0.2,0.3,0.4]
tftcphys=[(tfphy,tcphy) for tfphy in tfphys for tcphy in tcphys if tfphy>=2*tcphy]

labels=['lbd']+['2st2step_SYM','2st2step_SYMshare','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']

def run(tfphy,tcphy):
    label2res={}
    fig,axs=yu.getFigAxs(1,len(labels),sharex=True,sharey=True)
    axs[0,0].set_ylim(ylim)
    yu.addColHeader(axs,labels)
    for ilabel,label in enumerate(labels):
        ax=axs[0,ilabel]
        
        ens2dat={}
        ens2fitlabel={}
        for ens in enss:
            m,e=dic_sgm_ChPT[(yu.ens2label[ens],'ratio_N3LO')]
            factor=yu.ens2amul[ens]*yu.ens2aInv[ens]*yu.jackknife_pseudo(m,e,ens2Njk[ens])[:,0]
            fits=yu.getFits(f'{g}_{ens}_{label}{app}')
            ind=np.argmin([np.abs(tfmin-tfphy/yu.ens2a[ens]) + np.abs(tcmin - tcphy/yu.ens2a[ens] ) for (tfmin,tcmin),*_ in fits])
            fit=fits[ind]
            ens2fitlabel[ens]=fit[0]
            
            dat=fit[1][:,0]*factor
            ens2dat[ens]=dat
            
            mean,err=yu.jackme(dat)
            ax.errorbar(yu.ens2a[ens]**2,mean,err,color='r')
        
        if label in ['2st2step_SYMshare']:
            if tfphy in [0.8] and tcphy in [0.2,0.3]:
                print(tfphy,tcphy,ens2fitlabel)
        
        fits=yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yuc.lat_a2s_plt)

        pars_jk,probs_jk=yu.jackMA(fits)
        mean,err=yu.jackme(pars_jk)
        x=yuc.lat_a2s_plt; ymin=mean-err; ymax=mean+err
        ax.plot(x,mean,color='r',linestyle='--',marker='')
        ax.fill_between(x, ymin, ymax, color='r', alpha=0.1,label=f'{yu.un2str(mean[0],err[0])}')
        ax.legend()
        
        label2res[label]=pars_jk[:,0]
    yu.finalizePlot(f'CEs{app}/{tfphy}_{tcphy}',mkdirQ=True)
    return label2res

fig,axs=yu.getFigAxs(1,len(labels),sharex=True,sharey=True)
axs[0,0].set_ylim(ylim)
yu.addColHeader(axs,labels)
xunit=1; yunit=1
for tfmin,tcmin in tftcphys:
    label2res=run(tfmin,tcmin)
    itcmin=tcphys.index(tcmin)
    shift_fit=0
    for ilabel,label in enumerate(labels):
        ax=axs[0,ilabel]
        
        res=label2res[label]
        
        mean,err=yu.jackme(res)
        plt_x=(tfmin+itcmin*0.01+shift_fit)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[itcmin%8],fmt=yu.fmts8[itcmin%8])

yu.finalizePlot(f'CEs{app}')

0.8 0.2 {'b': (10, 3), 'c': (12, 3), 'd': (14, 4), 'e': (17, 4)}
0.8 0.3 {'b': (10, 4), 'c': (12, 4), 'd': (14, 5), 'e': (17, 6)}
